In [1]:
# !pip install transformers datasets seqeval evaluate torch
!pip install kiwipiepy
!pip install --upgrade torch torchvision
!pip install transformers seqeval[gpu]
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 17.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 13.5 MB/s eta 0:00:00
  Created wheel for kiwipiepy_model: filename=kiwipiepy_model-0.20.0-py3-none-any.whl size=34818026 sha256=108af97cb5318b4ec92c466084e730793f0f34d890f67e432ce37c5da635d847
  Stored in directory: /root/.cache/pip/wheels/ca/c8/52/3a539d6e9065b191fe1c215e0203dcc3e00601c0e3d3d39824
Successfully built kiwipiepy_model
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━

In [2]:
# import json
# from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, AutoModel
# from datasets import Dataset, DatasetDict
import sqlite3
# import pandas as pd
import ast
import re
# import evaluate
# import torch
# import torch.nn as nn
# import numpy as np
# import seqeval.metrics


import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertConfig, BertForTokenClassification

from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords

from torch.utils.data import Dataset

##tech_dict 불러오기

In [3]:
# 데이터베이스 파일 경로 설정
db_path = 'asia.db'

# 딕셔너리 초기화
tech_dict = {
    "languages": [],
    "frameworks": [],
    "libraries": [],
    "tools": []
}

# SQLite 데이터베이스 연결
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 기술 요소 불러오기
    query = "SELECT category, name FROM technical_element"
    cursor.execute(query)
    rows = cursor.fetchall()

    # 딕셔너리 변환
    for row in rows:
        category = row[0].strip().lower()
        name = row[1].strip()

        if category == "language":
            tech_dict["languages"].append(name)
        elif category == "framework":
            tech_dict["frameworks"].append(name)
        elif category == "library":
            tech_dict["libraries"].append(name)
        elif category == "tool":
            tech_dict["tools"].append(name)

    # 중복 제거 및 정렬 (선택 사항)
    for key in tech_dict:
        tech_dict[key] = sorted(list(set(tech_dict[key])))

    # 출력
    print("기술 사전 변환 완료:")
    print(tech_dict)

except sqlite3.Error as e:
    print(f"SQLite 에러 발생: {e}")
finally:
    if conn:
        conn.close()


기술 사전 변환 완료:
{'languages': ['abap', 'ada', 'bash', 'c', 'c#', 'c++', 'css', 'dart', 'elixir', 'erlang', 'f#', 'fortran', 'go', 'groovy', 'haskell', 'hiveql', 'html', 'java', 'javascript', 'julia', 'kotlin', 'lua', 'matlab', 'nodejs', 'perl', 'php', 'python', 'r', 'rescript', 'ruby', 'rust', 'scala', 'solidity', 'swift', 'typescript', 'vhdl'], 'frameworks': ['angular', 'appium', 'armeria', 'backbonejs', 'codeigniter', 'dagger', 'django', 'dropwizard', 'echo', 'electra', 'electron', 'emberjs', 'expressjs', 'falcon', 'fastapi', 'fastify', 'fiber', 'flask', 'flink', 'gatsby', 'grpc', 'hadoop', 'jasmine', 'javafx', 'junit', 'kotest', 'ktor', 'laravel', 'meteor', 'mocha', 'mockito', 'mono', 'nestjs', 'netty', 'nextjs', 'nuxtjs', 'phoenix', 'ray', 'react native', 'reactjs', 'reactorkit', 'relay', 'ribs', 'ruby on rails', 'sanic', 'selenium', 'spark', 'spring', 'springboot', 'svelte', 'swagger', 'tailwind', 'thrift', 'vuejs'], 'libraries': ['alamofire', 'apollo', 'beautifulsoup', 'bokeh', 'emo

##동의어를 대표어로 변환하고 토큰화하는 과정("2025-01-24-12_categorized.csv" 결과 파일을 가지고 있으므로 생략)


In [ ]:
# # CSV 파일 불러오기
# df = pd.read_csv(f'./2025-01-24-12_final.csv')

# # 'description'과 'requirement' 열 합치기 및 문자열로 변환
# df['combined_text'] = df[['technicalTags', 'description', 'requirement']].fillna('').astype(str).agg(' '.join, axis=1)
# df['preferredExperience'] = df['preferredExperience'].fillna('')

# df['combined_text'] = df['combined_text'].apply(lambda cell: cell.replace(".", "").replace('/', ', ').lower() if isinstance(cell, str) else cell)
# df['preferredExperience'] = df['preferredExperience'].apply(lambda cell: cell.replace(".", "").replace('/', ', ').lower() if isinstance(cell, str) else cell)

# kiwi = Kiwi(typos='basic_with_continual_and_lengthening')

# # 불용어 설정
# stop_words = Stopwords()

# # 파일 경로 설정
# file_path = "./technicalTags.txt"

# # 파일에서 단어 읽기
# with open(file_path, "r", encoding="utf-8") as f:
#     words = f.read().splitlines()

# words_remove = [word.replace('.', '').lower() for word in words]

# # 사용자 사전에 단어 추가
# for word in words_remove:
#     kiwi.add_user_word(word, "NNG")

# # SQLite 데이터베이스에서 기술 카테고리 데이터 로드
# conn = sqlite3.connect("./asia.db")
# cursor = conn.cursor()
# cursor.execute("SELECT category, name, synonym FROM technical_element")
# rows = cursor.fetchall()
# conn.close()

# # 데이터 정리
# categories = {}
# all_incorrect_list = []
# replacement_map = {}

# for category, name, synonym in rows:
#     # category가 "language"이면 "it_language"로 변경
#     if category.lower().strip() == "language":
#         category = "it_language"

#     synonyms = synonym.split(",")  # ','로 분리하여 리스트로 저장
#     all_incorrect_list.extend(synonyms)  # 모든 동의어 리스트 저장

#     if category not in categories:
#         categories[category] = {}

#     # name 자체도 정규화된 이름으로 포함
#     categories[category][name] = name

#     for syn in synonyms:
#         replacement_map[syn.lower()] = name.lower().strip()  # 동의어 -> 올바른 명칭 매핑
#         categories[category][syn] = name

# # 사용자 사전에 단어 추가
# for incorrec in all_incorrect_list:
#     kiwi.add_user_word(incorrec, "NNG")

# # 영어 여부 확인 함수
# def is_english(word):
#     return re.match(r'^[a-zA-Z0-9#+\-\s]+$', word) is not None

# # 형태소 분석 함수
# def analyze_text(text):
#     if not isinstance(text, str):
#         return []
#     morphemes_kiwi = kiwi.tokenize(text, normalize_coda=True, stopwords=stop_words, split_complex=True)
#     return [morph for morph, pos, _, _ in morphemes_kiwi if pos.startswith('N') or pos == 'SL']

# # 형태소 리스트에서 영어 단어만 필터링
# def filter_english_words(morphemes):
#     if not isinstance(morphemes, list):
#         return []
#     return [word for word in morphemes if is_english(word)]

# # 오탈자 수정 함수
# def replace_typos(tokens, replacement_map):
#     if not isinstance(tokens, list):
#         return []
#     return [replacement_map.get(token.lower(), token) for token in tokens]

# # 형태소 분석 및 영어 필터링
# df['morpheme'] = df['combined_text'].apply(analyze_text)
# df['pre_morpheme'] = df['preferredExperience'].apply(analyze_text)

# df['morpheme_eng'] = df['morpheme'].apply(filter_english_words)
# df['pre_morpheme_eng'] = df['pre_morpheme'].apply(filter_english_words)

# df['morpheme_eng'] = df['morpheme_eng'].apply(lambda x: replace_typos(x, replacement_map))
# df['pre_morpheme_eng'] = df['pre_morpheme_eng'].apply(lambda x: replace_typos(x, replacement_map))

# # 데이터 전처리: 소문자로 변환, 공백 및 '.' 제거
# def preprocess(word):
#     return word.lower().replace(".", "").replace(" ", "")

# # 치환 및 분류 함수
# def classify_and_replace(tokens, categories):
#     if not isinstance(tokens, list):
#         return []
#     result = {key: set() for key in categories.keys()}
#     for token in tokens:
#         normalized_token = preprocess(token)
#         for category, mapping in categories.items():
#             if normalized_token in mapping:
#                 result[category].add(mapping[normalized_token])
#     return result

# # 각 카테고리에 대해 열 생성
# for category in categories.keys():
#     df[f'{category}'] = df['morpheme_eng'].apply(
#         lambda x: ", ".join(sorted(classify_and_replace(x, categories)[category]))
#     )
#     df[f'pre_{category}'] = df['pre_morpheme_eng'].apply(
#         lambda x: ", ".join(sorted(classify_and_replace(x, categories)[category]))
#     )

# # 학력 결정 함수
# def determine_degree(tokens):
#     if not isinstance(tokens, list):
#         tokens = []
#     if '학사' in tokens or '대졸' in tokens:
#         return 1
#     for i, token in enumerate(tokens):
#         if token in ['대학교', '대학']:
#             if any(next_token in ['졸업자', '졸업'] for next_token in tokens[i + 1:i + 11]):
#                 return 1
#     if '석사' in tokens:
#         return 2
#     elif '석' in tokens:
#         for i, token in enumerate(tokens[:-1]):
#             if token == '석' and tokens[i + 1] == '박사':
#                 return 2
#     if '박사' in tokens:
#         return 2
#     return 0

# # 어학 조건 결정 함수
# def determine_language(tokens):
#     if not isinstance(tokens, list):
#         tokens = []
#     language_map = {'어학': 1, '외국어': 1, '영어': 1, '일본어': 2, '일어': 2, '중국어': 3, '중어': 3}
#     for token in tokens:
#         if token in language_map:
#             return language_map[token]
#     return None

# # 학력 열 생성
# df['degree'] = df['morpheme'].apply(determine_degree)

# # 어학 열 생성
# df['language'] = df['morpheme'].apply(determine_language)

# # 필요한 열 정리 및 저장
# col = ['text', 'id', 'url', 'title', 'location', 'duty', 'degree', 'language',
#        'countryCode', 'company_name', 'crawling_dt', 'career',
#        'combined_text','morpheme', 'morpheme_eng',
#        'it_language', 'framework', 'library', 'tool',
#        'preferredExperience', 'pre_morpheme', 'pre_morpheme_eng',
#        'pre_it_language', 'pre_framework', 'pre_library', 'pre_tool',
#        ]

# df[col].to_csv(f'./2025-01-24-12_categorized.csv', index=False, encoding='utf-8-sig')

## 영어토큰 리스트 불러오기(input값)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
### 작업 경로 이동
%cd /content/drive/MyDrive/ai

/content/drive/MyDrive/ai


In [4]:
# CSV 파일 불러오기
df = pd.read_csv(f'./2025-01-24-12_categorized.csv')
# morpheme_eng 열의 값만 가져오기
tokens_list = df['morpheme_eng']
english_tokens_list = []
for row in tokens_list:
  row = ast.literal_eval(row)
  english_tokens_list.append(row)

for row in english_tokens_list:
  print(row)


['python', 'net', 'c#', 'ai', 'ai', 'data', 'camp', 'ui', 'mlops', 'platform', 'lisa', 'look', 'in', 'smart', 'with', 'ai', 'ai', 'ui', 'ux', 'ai', 'ai', 'ai', 'smart', 'factory', 'solution', 'ai', 'ai', 'ai', 'smart', 'factory', 'solution', 'ai', 'ai', 'anomaly', 'detection', 'semi', 'supervised', 'self', 'supervised', 'video', 'vision', 'classification', 'segmentation', 'object', 'detection', 'ai']
['java', 'javascript', 'mysql', 'spring', 'kisa', 'spring framework', 'springboot']
['nodejs', 'typescript', 'javascript', 'firebase', 'spring', 'aws cloud9', 'azure', 'nosql', 'llm', 'test', 'code', 'javascript', 'typescript', 'api', 'django', 'flask', 'ruby on rails', 'spring', 'aws athena', 'azure', 'aliyun', 'dau', 'nodejs']
['a', 'b', 'mmm', 'mta', 'casual', 'impact', 'sql', 'python', 'bi', 'tableau', 'redash']
['emr', 'electric', 'medical', 'record', 'windows', 'client', 'c#', 'wpf', 'c#', 'wpf', 'mvvm', 'pattern', 'xaml', 'ui', 'ui', 'library', 'devexpress', 'telerik', 'custom', 'co

## 어노테이션(BIO 태그 적용 전,후)


In [119]:
import time
#가지고 있는 기술사전에 대하여 수동 어노테이션
def create_data(english_tokens_list, tech_dict):
    data = []
    for index, tokens in enumerate(english_tokens_list):
      #morphs_eng가 [] 인경우 예외처리
      if len(tokens) == 0:
        data.append({
            "index" : index,
            "tokens": [],
          "tags": []
        })
      #[]이 아닌 경우
      else :
        tags = []
        for token in tokens:
          tag = "O" # 기본 태그를 'O'로 설정
          for category, tech_names in tech_dict.items():
            if token in tech_names:
              tag = "B-"+category.upper() # 카테고리 이름으로 태그 설정 (대문자)
          tags.append(tag)
        data.append({
            "index" : index,
            "tokens": tokens,
            "tags": tags
        })
    return data

print("-----------------------------------------")
#data에 I- 태그 적용
def update_ner_tags(data):
    updated_data = []
    for entry in data:
        new_tokens = []
        new_tags = []
        for token, tag in zip(entry["tokens"], entry["tags"]):
            token_parts = token.split()
            if len(token_parts) > 1:  # 띄어쓰기가 있는 경우
                new_tokens.extend(token_parts)
                new_tags.append(tag)  # 첫 번째 단어는 B- 태그 유지
                i_tag = tag.replace("B-", "I-") if tag.startswith("B-") else tag
                new_tags.extend([i_tag] * (len(token_parts) - 1))  # 이후 단어는 I- 태그로 변경
            else:
                new_tokens.append(token)
                new_tags.append(tag)
        updated_data.append({"index": entry["index"], "tokens": new_tokens, "tags": new_tags})
    return updated_data



# 실행 시간 측정을 위한 코드 추가
start_time = time.time()

# 수동 어노테이션 데이터 생성
data = create_data(english_tokens_list, tech_dict)
# print(data)
# I-태그 변환 실행
updated_ner_data = update_ner_tags(data)


# 실행 종료 시간 기록
end_time = time.time()
# 실행 시간 출력
execution_time = end_time - start_time


# 결과 출력 (data)
# for sample in updated_ner_data:
#     print("    {")
#     print(f'        "index": {sample["index"]},')
#     print(f'        "tokens": {sample["tokens"]},')
#     print(f'        "tags": {sample["tags"]}')
#     print("    },")
print(f"⏳ 실행 시간: {execution_time:.6f} 초")

-----------------------------------------
⏳ 실행 시간: 1.160939 초


##어노테이션 결과 csv파일로 저장(생략가능)

In [ ]:
# CSV 파일로 저장
# rows = []
# for original, updated in zip(data, updated_ner_data):
#     rows.append({
#         "index": original["index"],
#         "original_tokens": ', '.join(original["tokens"]),
#         "original_tags": ', '.join(original["tags"]),
#         "updated_tokens": ', '.join(updated["tokens"]),
#         "updated_tags": ', '.join(updated["tags"])
#     })

# df = pd.DataFrame(rows)
# df.to_csv("BIO_tags_comparison.csv", index=False)

# # 결과 출력
# print("CSV 파일 저장 완료: BIO_tags_comparison.csv")
# CSV 파일로 저장
rows = []
for original, updated in zip(data, updated_ner_data):
    rows.append({
        "index": original["index"],
        "original_tokens": json.dumps(original["tokens"]),  # 리스트를 문자열로 변환
        "original_tags": json.dumps(original["tags"]),
        "updated_tokens": json.dumps(updated["tokens"]),
        "updated_tags": json.dumps(updated["tags"])
    })

df = pd.DataFrame(rows)
df.to_csv("BIO_tags_comparison.csv", index=False)

# 결과 출력
print("CSV 파일 저장 완료: BIO_tags_comparison.csv")

CSV 파일 저장 완료: BIO_tags_comparison.csv


In [ ]:
# # CSV 파일 불러오기
# df = pd.read_csv(f'./BIO_tags_comparison.csv')
# # morpheme_eng 열의 값만 가져오기
# updated_tokens = df['updated_tokens'].apply(json.loads)
# updated_tags = df['updated_tags'].apply(json.loads)
# tokens_list = []
# tags_list = []
# for updated_tokens_list in updated_tokens:
#   temp_list = []
#   for row in updated_tokens_list:
#     temp_list.append(row)
#   tokens_list.append(temp_list)

# for updated_tags_list in updated_tags:
#   temp_list = []
#   for row in updated_tags_list:
#     temp_list.append(row)
#   tags_list.append(temp_list)


In [7]:
#어노테이션 태그 선언
labels_to_id = {
    "O": 0,
    "B-LANGUAGES": 1,
    "B-FRAMEWORKS": 2,
    "B-LIBRARIES": 3,
    "B-TOOLS": 4,
    "I-LANGUAGES": 5,
    "I-FRAMEWORKS": 6,
    "I-LIBRARIES": 7,
    "I-TOOLS": 8
}
id_to_labels = {v: k for k, v in labels_to_id.items()}
labels_list = list(labels_to_id.keys())
print(id_to_labels)
num_labels = len(labels_list)


{0: 'O', 1: 'B-LANGUAGES', 2: 'B-FRAMEWORKS', 3: 'B-LIBRARIES', 4: 'B-TOOLS', 5: 'I-LANGUAGES', 6: 'I-FRAMEWORKS', 7: 'I-LIBRARIES', 8: 'I-TOOLS'}


In [8]:
data = pd.DataFrame(updated_ner_data)  # data가 리스트일 경우 DataFrame으로 변환
print(data)

      index                                             tokens  \
0         0  [python, net, c#, ai, ai, data, camp, ui, mlop...   
1         1  [java, javascript, mysql, spring, kisa, spring...   
2         2  [nodejs, typescript, javascript, firebase, spr...   
3         3  [a, b, mmm, mta, casual, impact, sql, python, ...   
4         4  [emr, electric, medical, record, windows, clie...   
...     ...                                                ...   
3944   3944  [git, reactjs, css, html, javascript, aws, ath...   
3945   3945  [html, java, kotlin, aws, athena, ux, spring, ...   
3946   3946  [git, github, reactjs, css, javascript, typesc...   
3947   3947  [git, graphql, reactjs, react, native, reactjs...   
3948   3948  [github, pytorch, reactjs, tensorflow, c, c++,...   

                                                   tags  
0     [B-LANGUAGES, O, B-LANGUAGES, O, O, O, O, O, O...  
1     [B-LANGUAGES, B-LANGUAGES, B-TOOLS, B-FRAMEWOR...  
2     [B-LANGUAGES, B-LANGUAGES, 

In [9]:
data1 = data.drop(columns=['index'])
data1.loc[:, 'tokens'] = data1.loc[:,'tokens'].map(lambda x:' '.join(x))
data1.loc[:, 'tags'] = data1.loc[:,'tags'].map(lambda x:','.join(x))
data1

,tokens,tags
0,python net c# ai ai data camp ui mlops platfor...,"B-LANGUAGES,O,B-LANGUAGES,O,O,O,O,O,O,O,O,O,O,..."
1,java javascript mysql spring kisa spring frame...,"B-LANGUAGES,B-LANGUAGES,B-TOOLS,B-FRAMEWORKS,O..."
2,nodejs typescript javascript firebase spring a...,"B-LANGUAGES,B-LANGUAGES,B-LANGUAGES,O,B-FRAMEW..."
3,a b mmm mta casual impact sql python bi tablea...,"O,O,O,O,O,O,O,B-LANGUAGES,O,B-TOOLS,B-TOOLS"
4,emr electric medical record windows client c# ...,"O,O,O,O,O,O,B-LANGUAGES,O,B-LANGUAGES,O,O,O,O,..."
...,...,...
3944,git reactjs css html javascript aws athena spr...,"B-TOOLS,B-FRAMEWORKS,B-LANGUAGES,B-LANGUAGES,B..."
3945,html java kotlin aws athena ux spring framewor...,"B-LANGUAGES,B-LANGUAGES,B-LANGUAGES,B-TOOLS,I-..."
3946,git github reactjs css javascript typescript s...,"B-TOOLS,B-TOOLS,B-FRAMEWORKS,B-LANGUAGES,B-LAN..."
3947,git graphql reactjs react native reactjs react...,"B-TOOLS,B-LIBRARIES,B-FRAMEWORKS,B-FRAMEWORKS,..."


In [10]:
#하이퍼파라미터
from transformers import BertTokenizerFast, BertTokenizer
model_name = "bert-base-uncased" # 사용하는 모델 이름
tokenizer = BertTokenizerFast.from_pretrained(model_name) # BertTokenizer 직접 지정 (Python 기반)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [11]:
class dataset(Dataset):
  def __init__(self, dataframe, tokenizer, max_len):
        self.len = len(dataframe)
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

  def __getitem__(self, index):
        # step 1: get the sentence and word labels
        tokens = self.data.tokens[index].strip().split()
        tags = self.data.tags[index].split(",")

        # step 2: use tokenizer to encode sentence (includes padding/truncation up to max length)
        # BertTokenizerFast provides a handy "return_offsets_mapping" functionality for individual tokens
        encoding = self.tokenizer(tokens,
                                  is_split_into_words=True,
                                  return_offsets_mapping=True,
                                  padding='max_length',
                                  truncation=True,
                                  max_length=self.max_len)

        # step 3: create token labels only for first word pieces of each tokenized word
        labels = [labels_to_id.get(tag, labels_to_id['O']) for tag in tags]
        # code based on https://huggingface.co/transformers/custom_datasets.html#tok-ner
        # create an empty array of -100 of length max_length
        encoded_labels = np.ones(len(encoding["offset_mapping"]), dtype=int) * -100

        # set only labels whose first offset position is 0 and the second is not 0
        i = 0
        for idx, mapping in enumerate(encoding["offset_mapping"]):
          if mapping[0] == 0 and mapping[1] != 0:
            # overwrite label
            encoded_labels[idx] = labels[i]
            i += 1

        # step 4: turn everything into PyTorch tensors
        item = {key: torch.as_tensor(val) for key, val in encoding.items()}
        item['labels'] = torch.as_tensor(encoded_labels)

        return item

  def __len__(self):
        return self.len

In [12]:
### NER 모델 생성
MAX_LEN = 128
TRAIN_BATCH_SIZE = 4
VALID_BATCH_SIZE = 2
EPOCHS = 3
LEARNING_RATE = 1e-05
MAX_GRAD_NORM = 10


from transformers import BertTokenizer, BertTokenizerFast, AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=len(labels_to_id))
model.to(device)

Using device: cuda


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [13]:
# 데이터 분리
train_size = 0.8
val_size = 0.1  # 검증 데이터 비율
test_size = 0.1  # 테스트 데이터 비율

# 1️⃣ Train 데이터 샘플링 (80%)
train_dataset = data1.sample(frac=train_size, random_state=200)
remaining_data = data1.drop(train_dataset.index).reset_index(drop=True)

# 2️⃣ Validation 데이터 샘플링 (10%)
val_dataset = remaining_data.sample(frac=val_size / (val_size + test_size), random_state=200)

# 3️⃣ Test 데이터는 남은 데이터 (10%)
test_dataset = remaining_data.drop(val_dataset.index).reset_index(drop=True)

# 인덱스 재정렬
train_dataset = train_dataset.reset_index(drop=True)
val_dataset = val_dataset.reset_index(drop=True)
test_dataset = test_dataset.reset_index(drop=True)

# 데이터 크기 확인
print("FULL Dataset: {}".format(data1.shape))
print('-'*80)
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("VALIDATION Dataset: {}".format(val_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))

# 데이터셋 생성
training_set = dataset(train_dataset, tokenizer, MAX_LEN)
validation_set = dataset(val_dataset, tokenizer, MAX_LEN)
testing_set = dataset(test_dataset, tokenizer, MAX_LEN)


FULL Dataset: (3949, 2)
--------------------------------------------------------------------------------
TRAIN Dataset: (3159, 2)
VALIDATION Dataset: (395, 2)
TEST Dataset: (395, 2)


In [ ]:
# 데이터 분리
# train_size = 0.8
# train_dataset = data1.sample(frac=train_size, random_state=200)
# test_dataset = data1.drop(train_dataset.index).reset_index(drop=True)
# train_dataset = train_dataset.reset_index(drop=True)

# print("FULL Dataset: {}".format(data.shape))
# print('-'*80)
# print("TRAIN Dataset: {}".format(train_dataset.shape))
# print('-'*80)
# print("TEST Dataset: {}".format(test_dataset.shape))

# # 데이터셋 생성
# training_set = dataset(train_dataset, tokenizer, MAX_LEN)
# testing_set = dataset(test_dataset, tokenizer, MAX_LEN)

FULL Dataset: (3949, 3)
--------------------------------------------------------------------------------
TRAIN Dataset: (3159, 2)
--------------------------------------------------------------------------------
TEST Dataset: (790, 2)


NameError: name 'MAX_LEN' is not defined

In [14]:
# 첫 번째 데이터 확인
print(training_set[0])

{'input_ids': tensor([  101, 13045, 22578,  4127, 23235,  9089, 22578,  6112,  1048,  5244,
         2003,  5244, 25222,  2361, 13045, 22578,  8241,  3075,  7396,  4646,
         4127, 23235,  9089, 22578,  2828,  2953,  2213,  2003,  5244, 27804,
         2361,   102,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

In [49]:
for token, label in zip(tokenizer.convert_ids_to_tokens(training_set[0]["input_ids"]), training_set[0]["labels"]):
  print('{0:10}  {1}'.format(token, label))

[CLS]       -100
node        1
##js        -100
types       1
##cript     -100
nest        2
##js        -100
cloud       0
l           0
##ms        -100
is          0
##ms        -100
cas         0
##p         -100
node        1
##js        -100
server      0
library     0
client      0
application  0
types       1
##cript     -100
nest        2
##js        -100
type        0
##or        -100
##m         -100
is          0
##ms        -100
csa         0
##p         -100
[SEP]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -100
[PAD]       -

In [16]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

val_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }
test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
validation_loader = DataLoader(validation_set, **val_params)
test_loader = DataLoader(testing_set, **test_params)

In [17]:
inputs = training_set[2]
input_ids = inputs["input_ids"].unsqueeze(0)
attention_mask = inputs["attention_mask"].unsqueeze(0)
labels = inputs["labels"].unsqueeze(0)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
labels = labels.to(device)

outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
initial_loss = outputs[0]
initial_loss

tensor(2.3513, device='cuda:0', grad_fn=<NllLossBackward0>)

In [18]:
tr_logits = outputs[1]
tr_logits.shape

torch.Size([1, 128, 9])

In [19]:
from transformers import AdamW

# 옵티마이저 정의
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Training 함수
def train(epoch):
    model.train()
    tr_loss, tr_accuracy = 0, 0
    nb_tr_examples, nb_tr_steps = 0, 0
    tr_preds, tr_labels = [], []

    for idx, batch in enumerate(training_loader):
        ids = batch['input_ids'].to(device, dtype=torch.long)
        mask = batch['attention_mask'].to(device, dtype=torch.long)
        labels = batch['labels'].to(device, dtype=torch.long)

        # Forward pass
        outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        tr_logits = outputs.logits
        tr_loss += loss.item()

        nb_tr_steps += 1
        nb_tr_examples += labels.size(0)

        if idx % 100 == 0:
            loss_step = tr_loss / nb_tr_steps
            print(f"Training loss per 100 training steps: {loss_step}")

        # Compute training accuracy
        flattened_targets = labels.view(-1)
        active_logits = tr_logits.view(-1, model.num_labels)
        flattened_predictions = torch.argmax(active_logits, axis=1)

        active_accuracy = labels.view(-1) != -100
        labels = torch.masked_select(flattened_targets, active_accuracy)
        predictions = torch.masked_select(flattened_predictions, active_accuracy)

        tr_labels.extend(labels.cpu().numpy())
        tr_preds.extend(predictions.cpu().numpy())

        tmp_tr_accuracy = accuracy_score(labels.cpu().numpy(), predictions.cpu().numpy())
        tr_accuracy += tmp_tr_accuracy

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=MAX_GRAD_NORM)

        # **✅ 옵티마이저 업데이트**
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    epoch_loss = tr_loss / nb_tr_steps
    tr_accuracy = tr_accuracy / nb_tr_steps
    print(f"Training loss epoch: {epoch_loss}")
    print(f"Training accuracy epoch: {tr_accuracy}")




/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [20]:
# Training 실행
for epoch in range(EPOCHS):
    print(f"Training epoch: {epoch + 1}")
    train(epoch)

Training epoch: 1
Training loss per 100 training steps: 2.21490478515625
Training loss per 100 training steps: 0.9457568673804255
Training loss per 100 training steps: 0.653082260593253
Training loss per 100 training steps: 0.506365593553728
Training loss per 100 training steps: 0.4163928982660063
Training loss per 100 training steps: 0.3548072361240457
Training loss per 100 training steps: 0.3130307602053028
Training loss per 100 training steps: 0.27947518466367316
Training loss epoch: 0.2546917463251967
Training accuracy epoch: 0.9311509504682342
Training epoch: 2
Training loss per 100 training steps: 0.10428103059530258
Training loss per 100 training steps: 0.04661391306631636
Training loss per 100 training steps: 0.045138360937102814
Training loss per 100 training steps: 0.04661283369463841
Training loss per 100 training steps: 0.045199201057346865
Training loss per 100 training steps: 0.04584203747485926
Training loss per 100 training steps: 0.0448536037160595
Training loss per 10

In [21]:
def valid(model, validation_loader):
    # put model in evaluation mode
    model.eval()

    eval_loss, eval_accuracy = 0, 0
    nb_eval_examples, nb_eval_steps = 0, 0
    eval_preds, eval_labels = [], []

    with torch.no_grad():
        for idx, batch in enumerate(validation_loader):

            ids = batch['input_ids'].to(device, dtype = torch.long)
            mask = batch['attention_mask'].to(device, dtype = torch.long)
            labels = batch['labels'].to(device, dtype = torch.long)

            outputs = model(input_ids=ids, attention_mask=mask, labels=labels)  # ✅ 올바른 반환값 저장
            loss = outputs.loss  # ✅ loss 가져오기
            eval_logits = outputs.logits  # ✅ logits 가져오기
            eval_loss += loss.item()  # ✅ 정상 작동

            nb_eval_steps += 1
            nb_eval_examples += labels.size(0)

            if idx % 100==0:
                loss_step = eval_loss/nb_eval_steps
                print(f"Validation loss per 100 evaluation steps: {loss_step}")

            # compute evaluation accuracy
            flattened_targets = labels.view(-1) # shape (batch_size * seq_len,)
            active_logits = eval_logits.view(-1, model.num_labels) # shape (batch_size * seq_len, num_labels)
            flattened_predictions = torch.argmax(active_logits, axis=1) # shape (batch_size * seq_len,)

            # only compute accuracy at active labels
            active_accuracy = labels.view(-1) != -100 # shape (batch_size, seq_len)

            labels = torch.masked_select(flattened_targets, active_accuracy)
            predictions = torch.masked_select(flattened_predictions, active_accuracy)

            eval_labels.extend(labels)
            eval_preds.extend(predictions)

            tmp_eval_accuracy = accuracy_score(labels.cpu().numpy(), predictions.cpu().numpy())
            eval_accuracy += tmp_eval_accuracy

    labels = [id_to_labels[id.item()] for id in eval_labels]
    predictions = [id_to_labels[id.item()] for id in eval_preds]

    eval_loss = eval_loss / nb_eval_steps
    eval_accuracy = eval_accuracy / nb_eval_steps
    print(f"Validation Loss: {eval_loss}")
    print(f"Validation Accuracy: {eval_accuracy}")

    return labels, predictions

In [22]:
labels, predictions = valid(model, validation_loader)

Validation loss per 100 evaluation steps: 0.0007541444501839578
Validation loss per 100 evaluation steps: 0.0212448082696001
Validation Loss: 0.02411836617705036
Validation Accuracy: 0.995221281290393


In [23]:
from seqeval.metrics import classification_report


# 기존의 labels, predictions는 1D 리스트이므로 2D 리스트로 변환
labels = [labels]  # ✅ 리스트 안에 리스트를 넣어서 변환
predictions = [predictions]  # ✅ 리스트 안에 리스트를 넣어서 변환

# 수정된 데이터로 평가 수행
print(classification_report(labels, predictions))

              precision    recall  f1-score   support

  FRAMEWORKS       1.00      0.98      0.99       518
   LANGUAGES       1.00      1.00      1.00      1083
   LIBRARIES       0.96      0.98      0.97       112
       TOOLS       0.98      0.99      0.99       956

   micro avg       0.99      0.99      0.99      2669
   macro avg       0.99      0.99      0.99      2669
weighted avg       0.99      0.99      0.99      2669



## 모델 저장

In [25]:
import os

directory = "./model"

if not os.path.exists(directory):
    os.makedirs(directory)

# save vocabulary of the tokenizer
tokenizer.save_vocabulary(directory)
# save the model weights and its configuration file
model.save_pretrained(directory)
print('All files saved')
print('This tutorial is completed')

All files saved
This tutorial is completed


## 테스트 데이터 평가

In [86]:
labels, predictions = valid(model, test_loader)

Validation loss per 100 evaluation steps: 0.036593254655599594
Validation loss per 100 evaluation steps: 0.02926527798212696
Validation Loss: 0.024466869357510027
Validation Accuracy: 0.9930461270489196


In [45]:
from seqeval.metrics import classification_report


# 기존의 labels, predictions는 1D 리스트이므로 2D 리스트로 변환
labels = [labels]  # ✅ 리스트 안에 리스트를 넣어서 변환
predictions = [predictions]  # ✅ 리스트 안에 리스트를 넣어서 변환

# 수정된 데이터로 평가 수행
print(classification_report(labels, predictions))

              precision    recall  f1-score   support

  FRAMEWORKS       0.99      0.96      0.98       570
   LANGUAGES       1.00      0.99      0.99      1021
   LIBRARIES       0.95      0.99      0.97       136
       TOOLS       0.98      0.99      0.99       931

   micro avg       0.99      0.99      0.99      2658
   macro avg       0.98      0.98      0.98      2658
weighted avg       0.99      0.99      0.99      2658



In [109]:
tokens_list = ''
for row in data1['tokens']:
  tokens_list = tokens_list + row
print(tokens_list)

python net c# ai ai data camp ui mlops platform lisa look in smart with ai ai ui ux ai ai ai smart factory solution ai ai ai smart factory solution ai ai anomaly detection semi supervised self supervised video vision classification segmentation object detection aijava javascript mysql spring kisa spring framework springbootnodejs typescript javascript firebase spring aws cloud9 azure nosql llm test code javascript typescript api django flask ruby on rails spring aws athena azure aliyun dau nodejsa b mmm mta casual impact sql python bi tableau redashemr electric medical record windows client c# wpf c# wpf mvvm pattern xaml ui ui library devexpress telerik custom control net framework client server restful gitb2b b2c aws athena docker kubernetes ci cd iac devops engineer aws athena docker kubernetes linux ci cd shell pythondjango python bi python django google slack rd party api api ci cdmission payroll agent python django google slack rd party api api ci cdswift ios ci cd dm ci cd unit 

In [126]:
#임의의 문장에 대해 테스트, 작동 잘 함
# 저장된 모델 및 토크나이저 경로
directory = "./model"

# 토크나이저 불러오기
tokenizer = BertTokenizerFast.from_pretrained(directory)

# 모델 불러오기
model = BertForTokenClassification.from_pretrained(directory)

# GPU 사용 가능하면 이동
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # 평가 모드 설정
# sentence = "mongodb and nosql are the same tool"
sentence = tokens_list


# 실행 시간 측정을 위한 코드 추가
start_time = time.time()

inputs = tokenizer(sentence.split(),
                    is_split_into_words=True,
                    return_offsets_mapping=True,
                    padding='max_length',
                    truncation=True,
                    max_length=MAX_LEN,
                    return_tensors="pt")

# move to gpu
ids = inputs["input_ids"].to(device)
mask = inputs["attention_mask"].to(device)
# forward pass
outputs = model(ids, attention_mask=mask)
logits = outputs[0]

active_logits = logits.view(-1, model.num_labels) # shape (batch_size * seq_len, num_labels)
flattened_predictions = torch.argmax(active_logits, axis=1) # shape (batch_size*seq_len,) - predictions at the token level

tokens = tokenizer.convert_ids_to_tokens(ids.squeeze().tolist())
token_predictions = [id_to_labels[i] for i in flattened_predictions.cpu().numpy()]
wp_preds = list(zip(tokens, token_predictions)) # list of tuples. Each tuple = (wordpiece, prediction)


# 실행 종료 시간 기록
end_time = time.time()

prediction = []
for token_pred, mapping in zip(wp_preds, inputs["offset_mapping"].squeeze().tolist()):
  #only predictions on first word pieces are important
  if mapping[0] == 0 and mapping[1] != 0:
    prediction.append(token_pred[1])
  else:
    continue



# 실행 시간 출력
execution_time = end_time - start_time
print(f"⏳ 실행 시간: {execution_time:.6f} 초")

# print(sentence.split())
# print(prediction)

⏳ 실행 시간: 0.648495 초
